In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
sys.path.append("/glade/u/home/islas/python/CESM2_with_CMIP7_aer/utils/")

### Output so4_a1 sfc emissions from scratch

This is necessary because the computation of the number is different for the agricultural and the shipping, solvents and waste components - a different diamater is assumed.  In Ben's files, these are already combined because the number was computed ahead of time.  So I will deal with the following separately:

- Agricultural (diameter 0.134 in both CMIP6 and CMIP7)
- Solvents + Waste (diameter of 0.261 in CMIP7, diameter of 0.134 in CMIP6)
- Shipping (diameter of 0.261 in both CMIP6 and CMIP7)

### Constants

In [2]:
avo = 6.02214076e23 # Avogadros constant
from molecular_weights import *

### Define output directory

In [3]:
outpath="/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/from_CEDS/CMIP6/"

### Read in the input4MIPS data

In [4]:
basepath="/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/input4MIPs/CMIP6/PNNL-JGCRI/"
so2 = xr.open_mfdataset(basepath+"SO2*.nc")

In [5]:
so2.sector.ids

'0: Agriculture; 1: Energy; 2: Industrial; 3: Transportation; 4: Residential, Commercial, Other; 5: Solvents production and application; 6: Waste; 7: International Shipping'

### Obtain the following separately: [Agricultural] [Solvents + Waste] [International]

In [6]:
so2_ag = so2.SO2_em_anthro.isel(sector=0, drop=True)
so2_slv_was = so2.SO2_em_anthro.isel(sector=5) + so2.SO2_em_anthro.isel(sector=6)
so2_ship = so2.SO2_em_anthro.isel(sector=7, drop=True)

### Conversion of SO2 mass to so4 molecules.  2.5% by moles of SO2 is emitted as so4

In [7]:
def so4_molecules_from_so2_mass(so2):

    # Convert from mass to moles (moles = mass (kg) * 1000 (g/kg) / molecular weight (g/mol)
    so2_moles = so2 * 1000. / mol_weights['SO2']

    # take 2.5 % of the moles for sulfate
    so4_moles = 0.025*so2_moles

    # Obtain molecules from moles using avogadros number (this is in per m2
    so4_molecules = so4_moles*avo

    # Convert to per cm2
    so4_molecules = so4_molecules / 1.e4
    
    return so4_molecules

In [8]:
so4_a1_ag = so4_molecules_from_so2_mass(so2_ag)
so4_a1_slv_was = so4_molecules_from_so2_mass(so2_slv_was)
so4_a1_ship = so4_molecules_from_so2_mass(so2_ship)

### Add attributes

In [9]:
so4_a1_ag.name="emiss"
so4_a1_ag.attrs["standard_name"] = "so4_a1_ag"
so4_a1_ag.attrs["long_name"] = "Emissions of so4_a1_ag"
so4_a1_ag.attrs["units"] = "molecules cm-2 s-1"
so4_a1_ag.attrs["molecular_weight"] = 115.
so4_a1_ag.attrs["molecular weight units"] = "g mole-1"
so4_a1_ag.attrs["sectors"] = "Agricultural"

so4_a1_slv_was.name = "emiss"
so4_a1_slv_was.attrs["standard_name"] = "so4_a1_slv_was"
so4_a1_slv_was.attrs["long_name"] = "Emissions of so4_a1_slv_was"
so4_a1_slv_was.attrs["units"] = "molecules cm-2 s-1"
so4_a1_slv_was.attrs["molecular_weight"] = 115.
so4_a1_slv_was.attrs["molecular_weight_units"] = "g mole-1"
so4_a1_slv_was.attrs["sectors"] = "Solventsproductionandapplication + Waste"

so4_a1_ship.name = "emiss"
so4_a1_ship.attrs["standard_name"] = "so4_a1_ship"
so4_a1_ship.attrs["long_name"] = "Emissions of so4_a1_ship"
so4_a1_ship.attrs["units"] = "molecules cm-2 s-1"
so4_a1_ship.attrs["molecular_weight"] = 115.
so4_a1_ship.attrs["molecular_weight_units"] = "g mole-1"
so4_a1_ship.attrs["sectors"] = "InternationalShipping"

In [10]:
dates = (so4_a1_ag.time.dt.year*1e5+so4_a1_ag.time.dt.month*100+so4_a1_ag.time.dt.day).astype('int32')
dates = xr.DataArray(dates, dims=['time'], coords=[so4_a1_ag.time], name='date')

In [11]:
so4_a1_ag_out = xr.merge([so4_a1_ag, dates])
so4_a1_ag_out.to_netcdf(outpath+'so4_a1_ag_emissions_native.nc')

so4_a1_slv_was_out = xr.merge([so4_a1_slv_was, dates])
so4_a1_slv_was_out.to_netcdf(outpath+'so4_a1_slv_was_emissions_native.nc')

so4_a1_ship_out = xr.merge([so4_a1_ship, dates])
so4_a1_ship_out.to_netcdf(outpath+'so4_a1_ship_emissions_native.nc')